# Solutions — Array Functions (Medium 12)

**Datasets:** `sales_transactions`, `sales_franchises`, `wanderbricks.reviews`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")
franchises   = spark.table("samples.bakehouse.sales_franchises")
reviews      = spark.table("samples.wanderbricks.reviews")

## Solution 1 — Collect Unique Products per Franchise

In [ ]:
result_1 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name")
    .agg(
        F.collect_set("product").alias("products_sold"),
        F.countDistinct("product").alias("product_count")
    )
    .orderBy("franchiseID")
)

## Solution 2 — Explode Products Array

In [ ]:
franchise_products = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name")
    .agg(F.collect_set("product").alias("products_sold"))
)
result_2 = (
    franchise_products
    .select("franchiseID","name", F.explode("products_sold").alias("product"))
    .orderBy("franchiseID","product")
)

## Solution 3 — Flag Franchises that Sell Coffee

In [ ]:
franchise_products = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name")
    .agg(
        F.collect_set("product").alias("products_sold"),
        F.countDistinct("product").alias("product_count")
    )
)
result_3 = (
    franchise_products
    .withColumn("sells_coffee", F.array_contains("products_sold", "Coffee"))
    .select("franchiseID","name","sells_coffee","product_count")
    .orderBy(F.col("sells_coffee").desc())
)

## Solution 4 — Array Set Operations (First Half vs Second Half Products)

In [ ]:
# Why: split by year-month midpoint; array_intersect/array_except operate on collected arrays
midpoint = transactions.agg(F.percentile_approx(F.unix_timestamp("dateTime").cast("double"), 0.5)).collect()[0][0]

first_half = (
    transactions
    .filter(F.unix_timestamp("dateTime") <= midpoint)
    .groupBy("franchiseID")
    .agg(F.collect_set("product").alias("first_half_products"))
)
second_half = (
    transactions
    .filter(F.unix_timestamp("dateTime") > midpoint)
    .groupBy("franchiseID")
    .agg(F.collect_set("product").alias("second_half_products"))
)
result_4 = (
    first_half
    .join(second_half, on="franchiseID")
    .withColumn("common_products",  F.array_intersect("first_half_products","second_half_products"))
    .withColumn("dropped_products", F.array_except("first_half_products","second_half_products"))
    .select("franchiseID","first_half_products","second_half_products","common_products","dropped_products")
)

## Solution 5 — Indexed Purchase History per Customer (posexplode)

In [ ]:
# Why: posexplode returns (pos, col) giving a zero-based purchase position
customer_products = (
    transactions
    .orderBy("customerID","dateTime")
    .groupBy("customerID")
    .agg(F.collect_list("product").alias("purchase_history"))
)
result_5 = (
    customer_products
    .select("customerID",
            F.posexplode("purchase_history").alias("purchase_position","product"))
    .orderBy("customerID","purchase_position")
)

## Solution 6 — Sorted Products & First Alphabetically per Franchise

In [ ]:
result_6 = (
    transactions
    .join(franchises, on="franchiseID")
    .groupBy("franchiseID","name")
    .agg(F.collect_set("product").alias("products"))
    .withColumn("sorted_products", F.sort_array("products"))
    .withColumn("first_alphabetical_product", F.element_at("sorted_products", 1))
    .select("franchiseID","name","sorted_products","first_alphabetical_product")
)

## Solution 7 — Flatten Monthly Comment Arrays per Property

In [ ]:
# Why: collect_list gathers per-property arrays; flatten merges them into one
monthly_comments = (
    reviews
    .filter(F.col("is_deleted") == False)
    .withColumn("month", F.date_trunc("month", "created_at"))
    .groupBy("property_id","month")
    .agg(F.collect_list("comment").alias("month_comments"))
)
result_7 = (
    monthly_comments
    .groupBy("property_id")
    .agg(F.collect_list("month_comments").alias("monthly_comment_arrays"))
    .withColumn("all_comments_flat", F.flatten("monthly_comment_arrays"))
    .withColumn("total_comment_count", F.size("all_comments_flat"))
    .orderBy(F.col("total_comment_count").desc())
)